### LGBM Regressor

We train a LGBM Regressor for each position to predict the number of points each player will score. We use `optuna` to find the best values for model hyperparameters using Bayesian Optimisation.

In [1]:
import pandas as pd

# add 'over_60_minutes' column
df = pd.read_csv(r"C:\Users\harve\OneDrive\Documents\Python\VScodeprojects\FPLbot\previous_seasons_dataset.csv")
df['over_60_minutes'] = (df['minutes'] >= 60).astype(int)

In [2]:
import optuna
from lightgbm import LGBMRegressor
import logging
import warnings
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial, position):
    
    data = df[(df['position'] == position) & (df['over_60_minutes'] == 1)]
    
    x = data[['team_market_value', 'opponent_market_value', 'value', 'was_home', 'points_last_game', 'total_points', 'mins_last_game',
              'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5', 'mean_mins_last_5', 'mean_points_last_10', 
              'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
              'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
              'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
              'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
              'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
              'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
              'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']] 
    
    y = data['points']

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    n_estimators = trial.suggest_int('n_estimators', 50, 2000)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    num_leaves = trial.suggest_int('num_leaves', 20, 300)
    max_depth = trial.suggest_int('max_depth', 3, 50)
    min_child_samples = trial.suggest_int('min_child_samples', 2, 200)
    subsample = trial.suggest_float('subsample', 0.3, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.3, 1.0)

    model = LGBMRegressor(n_estimators=n_estimators, learning_rate=learning_rate, num_leaves=num_leaves, max_depth=max_depth,
                              min_child_samples=min_child_samples, subsample=subsample, colsample_bytree=colsample_bytree, random_state=42, verbose=-1)

    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return rmse

def run_optimization(df, positions):
    best_params = {}
    for position in positions:
    
        study_lgbm = optuna.create_study(direction='minimize')
        study_lgbm.optimize(lambda trial: objective(trial, position), n_trials=500)
        best_params[(position)] = study_lgbm.best_params
        print(f"Best LGBM params for {position}: {study_lgbm.best_params}, RMSE: {study_lgbm.best_value}")

    return best_params

positions = ['GK', 'DEF', 'MID', 'FWD']
best_hyperparameters = run_optimization(df, positions)

c:\Users\harve\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Best LGBM params for GK: {'n_estimators': 87, 'learning_rate': 0.011106507786462324, 'num_leaves': 117, 'max_depth': 3, 'min_child_samples': 135, 'subsample': 0.6520088297191946, 'colsample_bytree': 0.3683836003860962}, RMSE: 2.801882846856975
Best LGBM params for DEF: {'n_estimators': 848, 'learning_rate': 0.012357895455639674, 'num_leaves': 295, 'max_depth': 43, 'min_child_samples': 2, 'subsample': 0.7373754269131323, 'colsample_bytree': 0.43454703152261576}, RMSE: 2.2390908397716647
Best LGBM params for MID: {'n_estimators': 172, 'learning_rate': 0.01774896695993906, 'num_leaves': 276, 'max_depth': 30, 'min_child_samples': 85, 'subsample': 0.31352790510946826, 'colsample_bytree': 0.6440668688082585}, RMSE: 3.03022899855466
Best LGBM params for FWD: {'n_estimators': 475, 'learning_rate': 0.018435817589826493, 'num_leaves': 143, 'max_depth': 3, 'min_child_samples': 106, 'subsample': 0.830966122531464, 'colsample_bytree': 0.31352867744766005}, RMSE: 3.5864431453446373


Goalkeepers:

In [3]:
GK_data = df[(df['position'] == 'GK') & (df['over_60_minutes'] == 1)]
GK_points_target = GK_data['points']
GK_points_features = GK_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(GK_points_features, GK_points_target, train_size=0.8, test_size=0.2)

best_GK_lgbm_params = best_hyperparameters[('GK')]

GK_lgbm = LGBMRegressor(n_estimators=best_GK_lgbm_params['n_estimators'], learning_rate=best_GK_lgbm_params['learning_rate'], 
                        num_leaves = best_GK_lgbm_params['num_leaves'], max_depth = best_GK_lgbm_params['max_depth'], 
                        min_child_samples = best_GK_lgbm_params['min_child_samples'], subsample = best_GK_lgbm_params['subsample'],
                        colsample_bytree = best_GK_lgbm_params['colsample_bytree'], random_state=42)

cv_scores = cross_val_score(GK_lgbm, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
GK_lgbm.fit(x_train, y_train)
y_pred = GK_lgbm.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = GK_lgbm.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': GK_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))
print(100*'-')

print(f'Cross validation scores: {np.abs(cv_scores)}')
print(f'Mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'RMSE: {RMSE}, R-squared: {R_squared}')

                              Feature  Importance
1               opponent_market_value          42
0                   team_market_value          37
38  total_opponent_points_last_season          33
37    total_team_conceded_last_season          25
2                               value          24
16            mean_team_points_last_3          23
22          mean_team_conceded_last_5          23
31      mean_opponent_conceded_last_3          21
34           total_points_last_season          20
36      total_team_points_last_season          19
----------------------------------------------------------------------------------------------------
Cross validation scores: [2.94716161 2.75501559 2.93472317 2.67845479 2.81957506]
Mean cross validation score: 2.8269860440537244
RMSE: 2.7877559883328953, R-squared: 0.01442515367849706


Defenders:

In [4]:
DEF_data = df[(df['position'] == 'DEF') & (df['over_60_minutes'] == 1)]
DEF_points_target = DEF_data['points']
DEF_points_features = DEF_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(DEF_points_features, DEF_points_target, train_size=0.8, test_size=0.2)

best_DEF_lgbm_params = best_hyperparameters[('DEF')]

DEF_lgbm = LGBMRegressor(n_estimators=best_DEF_lgbm_params['n_estimators'], learning_rate=best_DEF_lgbm_params['learning_rate'], 
                        num_leaves = best_DEF_lgbm_params['num_leaves'], max_depth = best_DEF_lgbm_params['max_depth'], 
                        min_child_samples = best_DEF_lgbm_params['min_child_samples'], subsample = best_DEF_lgbm_params['subsample'],
                        colsample_bytree = best_DEF_lgbm_params['colsample_bytree'], random_state=42)

cv_scores = cross_val_score(GK_lgbm, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
DEF_lgbm.fit(x_train, y_train)
y_pred = DEF_lgbm.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = DEF_lgbm.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': DEF_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))
print(100*'-')

print(f'Cross validation scores: {np.abs(cv_scores)}')
print(f'Mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'RMSE: {RMSE}, R-squared: {R_squared}')

                        Feature  Importance
7                    total_mins        9834
35       total_mins_last_season        9826
34     total_points_last_season        9819
5                  total_points        8324
12          mean_points_last_10        7799
13            mean_mins_last_10        7187
2                         value        6878
26  mean_opponent_points_last_3        6799
15            total_team_points        6748
14        team_points_last_game        6688
----------------------------------------------------------------------------------------------------
Cross validation scores: [3.11355646 3.07317429 3.04784964 3.00810433 2.99675825]
Mean cross validation score: 3.0478885945864076
RMSE: 2.34036626395043, R-squared: 0.43543741204729514


Midfielders:

In [5]:
MID_data = df[(df['position'] == 'MID') & (df['over_60_minutes'] == 1)]
MID_points_target = MID_data['points']
MID_points_features = MID_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(MID_points_features, MID_points_target, train_size=0.8, test_size=0.2)

best_MID_lgbm_params = best_hyperparameters[('MID')]

MID_lgbm = LGBMRegressor(n_estimators=best_MID_lgbm_params['n_estimators'], learning_rate=best_MID_lgbm_params['learning_rate'], 
                        num_leaves = best_MID_lgbm_params['num_leaves'], max_depth = best_MID_lgbm_params['max_depth'], 
                        min_child_samples = best_MID_lgbm_params['min_child_samples'], subsample = best_MID_lgbm_params['subsample'],
                        colsample_bytree = best_MID_lgbm_params['colsample_bytree'], random_state=42)

cv_scores = cross_val_score(MID_lgbm, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
MID_lgbm.fit(x_train, y_train)
y_pred = MID_lgbm.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = MID_lgbm.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': MID_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))
print(100*'-')

print(f'Cross validation scores: {np.abs(cv_scores)}')
print(f'Mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'RMSE: {RMSE}, R-squared: {R_squared}')

                           Feature  Importance
1            opponent_market_value         639
5                     total_points         614
2                            value         569
33  mean_opponent_conceded_last_10         555
34        total_points_last_season         530
0                team_market_value         504
32   mean_opponent_conceded_last_5         501
23      mean_team_conceded_last_10         492
35          total_mins_last_season         490
7                       total_mins         473
----------------------------------------------------------------------------------------------------
Cross validation scores: [2.88851413 2.98309675 2.82137668 3.05350369 3.03440913]
Mean cross validation score: 2.95618007437052
RMSE: 3.1087679055775803, R-squared: 0.09533831503472845


Forwards:

In [6]:
FWD_data = df[(df['position'] == 'FWD') & (df['over_60_minutes'] == 1)]
FWD_points_target = FWD_data['points']
FWD_points_features = FWD_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(FWD_points_features, FWD_points_target, train_size=0.8, test_size=0.2)

best_FWD_lgbm_params = best_hyperparameters[('FWD')]

FWD_lgbm = LGBMRegressor(n_estimators=best_FWD_lgbm_params['n_estimators'], learning_rate=best_FWD_lgbm_params['learning_rate'], 
                        num_leaves = best_FWD_lgbm_params['num_leaves'], max_depth = best_FWD_lgbm_params['max_depth'], 
                        min_child_samples = best_FWD_lgbm_params['min_child_samples'], subsample = best_FWD_lgbm_params['subsample'],
                        colsample_bytree = best_FWD_lgbm_params['colsample_bytree'], random_state=42)

cv_scores = cross_val_score(GK_lgbm, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
FWD_lgbm.fit(x_train, y_train)
y_pred = FWD_lgbm.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = FWD_lgbm.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': FWD_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))
print(100*'-')

print(f'Cross validation scores: {np.abs(cv_scores)}')
print(f'Mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'RMSE: {RMSE}, R-squared: {R_squared}')

                          Feature  Importance
26    mean_opponent_points_last_3         109
24          total_opponent_points          98
1           opponent_market_value          97
17        mean_team_points_last_5          95
29        total_opponent_conceded          85
2                           value          84
0               team_market_value          78
32  mean_opponent_conceded_last_5          77
8              mean_points_last_3          71
4                points_last_game          70
----------------------------------------------------------------------------------------------------
Cross validation scores: [3.66283246 3.76133226 3.42728884 3.56461562 3.36754282]
Mean cross validation score: 3.556722400526053
RMSE: 3.521547008073554, R-squared: 0.053181825856662646


In [7]:
import joblib

joblib.dump(GK_lgbm, 'GK_lgbm.pkl')
joblib.dump(DEF_lgbm, 'DEF_lgbm.pkl')
joblib.dump(MID_lgbm, 'MID_lgbm.pkl')
joblib.dump(FWD_lgbm, 'FWD_lgbm.pkl')

['FWD_lgbm.pkl']